In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project = Path.cwd().parent

df = pd.read_csv(
    project / "Data-raw" / "Telco-Customer-Churn-Cleaned.csv"
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Check missing values, duplicates, and data types

In [2]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nChurn distribution:")
print(df["Churn"].value_counts())

Missing values: 0
Duplicate customer IDs: 0

Data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


Validate the target and feature values

In [3]:
print("Churn values:", df["Churn"].unique())
print("Tenure range:", df["tenure"].min(), df["tenure"].max())

print("TotalCharges minimum:", df["TotalCharges"].min())

print("\nInternet service categories:")
print(df["InternetService"].value_counts())

Churn values: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Tenure range: 0 72
TotalCharges minimum: 0.0

Internet service categories:
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64


Separate features and target
In machine learning:
- X (features): Information used to predict churn.
- y (target): Whether the customer churned.

In [4]:
X = df.drop(columns=["customerID", "Churn"])

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (7043, 19)
y shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


Split the dataset into training and testing sets
We'll use an 80/20 split.

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (5634, 19)
Testing features: (1409, 19)
Training target: (5634,)
Testing target: (1409,)


Identify numerical and categorical features
Your dataset contains both numerical and categorical information.

Numerical features include tenure, MonthlyCharges, and TotalCharges.

Categorical features include Contract, InternetService, PaymentMethod, and the additional service subscriptions.

We'll also treat SeniorCitizen as categorical because its values represent two categories rather than a continuous measurement.

In [9]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


Feature engineering and preprocessing

Machine learning models such as Logistic Regression cannot directly process text values like Month-to-month or Fiber optic.

We need to convert categorical values into numerical representations.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import (
    SimpleImputer
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created!")

Preprocessing pipeline created!


SimpleImputer: Handles missing values if they occur in future input data.
StandardScaler: Standardizes numerical features.
OneHotEncoder: Converts categorical values into machine-readable columns.
ColumnTransformer: Applies the correct transformations to different columns.
Pipeline: Organizes preprocessing into a reproducible workflow.

Fit preprocessing on training data

Transform training and testing features

In [11]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Processed testing shape:",
    X_test_processed.shape
)

Processed training shape: (5634, 46)
Processed testing shape: (1409, 46)


Inspect the processed features

View encoded feature names

In [12]:
feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(feature_names))

print("\nFeature names:")
print(feature_names)

Total processed features: 46

Feature names:
['num__tenure' 'num__MonthlyCharges' 'num__TotalCharges'
 'cat__gender_Female' 'cat__gender_Male' 'cat__SeniorCitizen_0'
 'cat__SeniorCitizen_1' 'cat__Partner_No' 'cat__Partner_Yes'
 'cat__Dependents_No' 'cat__Dependents_Yes' 'cat__PhoneService_No'
 'cat__PhoneService_Yes' 'cat__MultipleLines_No'
 'cat__MultipleLines_No phone service' 'cat__MultipleLines_Yes'
 'cat__InternetService_DSL' 'cat__InternetService_Fiber optic'
 'cat__InternetService_No' 'cat__OnlineSecurity_No'
 'cat__OnlineSecurity_No internet service' 'cat__OnlineSecurity_Yes'
 'cat__OnlineBackup_No' 'cat__OnlineBackup_No internet service'
 'cat__OnlineBackup_Yes' 'cat__DeviceProtection_No'
 'cat__DeviceProtection_No internet service' 'cat__DeviceProtection_Yes'
 'cat__TechSupport_No' 'cat__TechSupport_No internet service'
 'cat__TechSupport_Yes' 'cat__StreamingTV_No'
 'cat__StreamingTV_No internet service' 'cat__StreamingTV_Yes'
 'cat__StreamingMovies_No' 'cat__StreamingMovies_

Save your preprocessing pipeline and data split
We will save the preprocessing pipeline for reuse during model development.
However, we should not train a model on the entire dataset before testing, and we should not fit preprocessing separately on the test set.

Save preprocessing artifacts

In [13]:
import joblib

models_dir = project / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    preprocessor,
    models_dir / "preprocessor.pkl"
)

print("Preprocessor saved successfully!")

Preprocessor saved successfully!


Save training and testing datasets

In [14]:
processed_dir = project / "Data-processed"
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(
    processed_dir / "X_train.csv",
    index=False
)

X_test.to_csv(
    processed_dir / "X_test.csv",
    index=False
)

y_train.to_csv(
    processed_dir / "y_train.csv",
    index=False
)

y_test.to_csv(
    processed_dir / "y_test.csv",
    index=False
)

print("Training and testing datasets saved!")

Training and testing datasets saved!


Verify your files

In [15]:
files_to_check = [
    models_dir / "preprocessor.pkl",
    processed_dir / "X_train.csv",
    processed_dir / "X_test.csv",
    processed_dir / "y_train.csv",
    processed_dir / "y_test.csv"
]

for file in files_to_check:
    print(file.name, ":", file.exists())

preprocessor.pkl : True
X_train.csv : True
X_test.csv : True
y_train.csv : True
y_test.csv : True
